In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)


In [5]:
df = pd.read_csv(r"C:\Users\AMAR\Downloads\job_postings_flat.csv")
print(df.shape)
df.head()


(478895, 17)


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills
0,Data Analyst,"Summer Internship -Data Analyst Intern, Risk M...","Marlborough, MA",via Boatingrevealed.com,"Full-time, Part-time, and Internship",False,"New York, United States",2024-01-01 00:00:01,False,True,United States,NaN,NaN,NaN,BJ's Wholesale Club,['excel'],{'analyst_tools': ['excel']}
1,Data Analyst,"Staff Data Analyst Operations, Infrastructure ...","Fremont, CA",via ClimateTechList,Full-time,False,"California, United States",2024-01-01 00:00:11,True,False,United States,NaN,NaN,NaN,Tesla,"['tableau', 'flow']","{'analyst_tools': ['tableau'], 'other': ['flow']}"
2,Data Analyst,Junior Data Analyst - Entry Level,"Waco, TX",via ZipRecruiter,Full-time and Part-time,False,"Texas, United States",2024-01-01 00:00:15,True,False,United States,NaN,NaN,NaN,Next Recruiting,NaN,NaN
3,Data Analyst,"Data Analyst/Engineer, Supply Chain Optimizati...","Austin, TX",via ClimateTechList,Internship,False,"Texas, United States",2024-01-01 00:00:18,False,False,United States,NaN,NaN,NaN,Tesla,['spring'],{'libraries': ['spring']}
4,Data Scientist,It analyst,"Tampa, FL",via Talent.com,Full-time,False,"Florida, United States",2024-01-01 00:00:29,True,False,United States,NaN,NaN,NaN,VirtualVocations,NaN,NaN


In [6]:
print("Rows, Columns:", df.shape)
df.columns


Rows, Columns: (478895, 17)


Index(['job_title_short', 'job_title', 'job_location', 'job_via', 'job_schedule_type', 'job_work_from_home',
       'search_location', 'job_posted_date', 'job_no_degree_mention', 'job_health_insurance', 'job_country',
       'salary_rate', 'salary_year_avg', 'salary_hour_avg', 'company_name', 'job_skills', 'job_type_skills'],
      dtype='object')

In [7]:
(df.isna().sum()
   .sort_values(ascending=False)
   .head(10))


salary_hour_avg      468819
salary_year_avg      458560
salary_rate          448484
job_type_skills       68870
job_skills            68870
job_schedule_type      7912
job_location           1373
job_country             321
company_name              7
job_via                   6
dtype: int64

In [8]:
df[["salary_year_avg", "salary_hour_avg", "salary_rate"]].describe()


,salary_year_avg,salary_hour_avg
count,20335.000000,10076.000000
mean,120587.934616,49.163118
std,48245.399071,25.607060
min,15000.000000,8.000000
25%,85975.000000,30.000000
50%,113250.000000,47.620003
75%,149589.000000,62.389999
max,920000.000000,250.000000


In [9]:
print("Unique countries:", df["job_country"].nunique())
print("Top job_title_short:\n", df["job_title_short"].value_counts().head(10))


Unique countries: 160
Top job_title_short:
 job_title_short
Data Engineer                128994
Data Analyst                 112866
Data Scientist                97664
Senior Data Engineer          30608
Business Analyst              28584
Software Engineer             23402
Senior Data Scientist         21731
Senior Data Analyst           15347
Machine Learning Engineer     12860
Cloud Engineer                 6839
Name: count, dtype: int64


In [10]:
data = df.copy()

data.columns = (data.columns
                .str.strip()
                .str.lower()
                .str.replace(" ", "_"))

data["job_posted_date"] = pd.to_datetime(data["job_posted_date"], errors="coerce")


data["posted_date"] = data["job_posted_date"].dt.date
data["posted_year"] = data["job_posted_date"].dt.year
data["posted_month"] = data["job_posted_date"].dt.month
data["posted_month_name"] = data["job_posted_date"].dt.strftime("%b")

data.head(2)


,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills,posted_date,posted_year,posted_month,posted_month_name
0,Data Analyst,"Summer Internship -Data Analyst Intern, Risk M...","Marlborough, MA",via Boatingrevealed.com,"Full-time, Part-time, and Internship",False,"New York, United States",2024-01-01 00:00:01,False,True,United States,NaN,NaN,NaN,BJ's Wholesale Club,['excel'],{'analyst_tools': ['excel']},2024-01-01,2024,1,Jan
1,Data Analyst,"Staff Data Analyst Operations, Infrastructure ...","Fremont, CA",via ClimateTechList,Full-time,False,"California, United States",2024-01-01 00:00:11,True,False,United States,NaN,NaN,NaN,Tesla,"['tableau', 'flow']","{'analyst_tools': ['tableau'], 'other': ['flow']}",2024-01-01,2024,1,Jan


In [11]:
data["work_mode"] = np.where(data["job_work_from_home"] == True, "Remote", "Onsite/Hybrid")
data["work_mode"].value_counts()


work_mode
Onsite/Hybrid    415900
Remote            62995
Name: count, dtype: int64

In [12]:

main_data = data.copy()

salary_data = data[data["salary_year_avg"].notna()].copy()

print("Main:", main_data.shape)
print("Salary dataset:", salary_data.shape)


Main: (478895, 22)
Salary dataset: (20335, 22)


In [13]:
def salary_bucket(x):
    if pd.isna(x):
        return "Unknown"
    elif x < 60000:
        return "Low"
    elif x < 120000:
        return "Medium"
    else:
        return "High"

salary_data["salary_bucket"] = salary_data["salary_year_avg"].apply(salary_bucket)

salary_data["salary_bucket"].value_counts()


salary_bucket
Medium    9937
High      9215
Low       1183
Name: count, dtype: int64

In [15]:
main_data.to_csv("main_job_postings_global.csv", index=False)
salary_data.to_csv("salary_job_postings_global.csv", index=False)

print(" Saved 2 files successfully!")


 Saved 2 files successfully!


In [16]:
main_data["job_skills"] = main_data["job_skills"].astype(str).str.lower()
salary_data["job_skills"] = salary_data["job_skills"].astype(str).str.lower()


In [17]:
skills_list = [
    "sql", "excel", "power bi", "tableau",
    "python", "r",
    "aws", "azure", "gcp",
    "spark", "hadoop",
    "snowflake", "bigquery", "databricks",
    "pandas", "numpy"
]


In [18]:
def has_skill(text, skill):
    if pd.isna(text):
        return 0
    return int(skill in text)

for sk in skills_list:
    col = sk.replace(" ", "_")
    main_data[col] = main_data["job_skills"].apply(lambda x: has_skill(x, sk))
    salary_data[col] = salary_data["job_skills"].apply(lambda x: has_skill(x, sk))

skill_cols = [sk.replace(" ", "_") for sk in skills_list]

main_data["skills_count"] = main_data[skill_cols].sum(axis=1)
salary_data["skills_count"] = salary_data[skill_cols].sum(axis=1)

main_data[skill_cols + ["skills_count"]].head()


,sql,excel,power_bi,tableau,python,r,aws,azure,gcp,spark,hadoop,snowflake,bigquery,databricks,pandas,numpy,skills_count
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [19]:
skill_demand = main_data[skill_cols].sum().sort_values(ascending=False)
skill_demand.head(10)


r             321642
sql           251597
python        244416
aws           100386
azure          93849
spark          86230
tableau        73560
excel          71807
power_bi       66183
databricks     42650
dtype: int64

In [20]:
skill_salary = {}

for col in skill_cols:
    avg_sal = salary_data.loc[salary_data[col] == 1, "salary_year_avg"].mean()
    skill_salary[col] = avg_sal

skill_salary_df = (
    pd.DataFrame(skill_salary.items(), columns=["skill", "avg_salary"])
      .sort_values("avg_salary", ascending=False)
)

skill_salary_df.head(10)


,skill,avg_salary
9,spark,138616.841850
13,databricks,135879.070448
10,hadoop,135531.331656
14,pandas,134421.973713
8,gcp,133578.556632
11,snowflake,133083.305372
6,aws,131982.016008
15,numpy,131292.168103
12,bigquery,131189.938305
4,python,130296.767452


In [21]:
main_data.to_csv("main_job_postings_global_skills.csv", index=False)
salary_data.to_csv("salary_job_postings_global_skills.csv", index=False)

print(" Saved with skill columns!")


 Saved with skill columns!
